# 🔎 GHOSTTRACE — Data Detective

**Discover unusual patterns, investigate suspicious records, and keep evidence close.**

This notebook is a research prototype for exploring CSV datasets. It checks for:
- Missing values and duplicate rows
- Potential statistical outliers using the IQR rule
- Anomalies using Isolation Forest
- Relationships between numeric features
- Downloadable investigation results

> **Important:** An anomaly is not proof of fraud or error. Review findings in context. Use synthetic or appropriately authorized data.

## 1. Install dependencies
Run this cell if the required libraries are not installed in your notebook environment.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib seaborn

## 2. Import libraries

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
print("GHOSTTRACE is ready.")

## 3. Load a dataset

Choose one option:
- **Option A:** Upload a CSV in Google Colab using the upload dialog.
- **Option B:** Set `CSV_PATH` to a CSV file on your computer.
- **Option C:** Run the next cell to create a synthetic demo dataset.

In [ ]:
# Option A: Google Colab upload
# Uncomment these lines if you are using Google Colab:
# from google.colab import files
# uploaded = files.upload()
# CSV_PATH = next(iter(uploaded))

# Option B: local file path, for example:
# CSV_PATH = "my_dataset.csv"

# Option C: set USE_DEMO = True to create sample data
USE_DEMO = True
CSV_PATH = None

if USE_DEMO:
    rng = np.random.default_rng(19)
    n = 250
    df = pd.DataFrame({
        "customer_age": rng.integers(18, 76, n),
        "monthly_spend": rng.normal(2500, 700, n).clip(100, None).round(2),
        "visits_per_month": rng.poisson(4, n),
        "support_tickets": rng.poisson(1, n),
    })
    # Add a few intentionally unusual synthetic observations for demonstration.
    df.loc[[12, 77, 184], "monthly_spend"] = [18000, 22000, 1]
    df.loc[[12, 77], "visits_per_month"] = [0, 1]
    df["customer_id"] = [f"C{i+1:04d}" for i in range(n)]
else:
    if CSV_PATH is None:
        raise ValueError("Set CSV_PATH to your CSV file path, or set USE_DEMO = True.")
    df = pd.read_csv(CSV_PATH)

print(f"Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
display(df.head(10))

## 4. Dataset overview

In [ ]:
overview = pd.DataFrame({
    "metric": ["Rows", "Columns", "Missing cells", "Duplicate rows"],
    "value": [
        len(df),
        len(df.columns),
        int(df.isna().sum().sum()),
        int(df.duplicated().sum())
    ]
})
display(overview)
display(df.dtypes.rename("data_type").to_frame())
display(df.describe(include="all").T)

## 5. Evidence board: missing values, duplicates, and potential outliers

In [ ]:
findings = []

missing = df.isna().sum()
for col, count in missing[missing > 0].items():
    pct = count / len(df) * 100
    findings.append({
        "type": "Missing data",
        "priority": "High" if pct >= 20 else "Medium" if pct >= 5 else "Low",
        "finding": f"{col} has {count:,} missing value(s) ({pct:.1f}%).",
        "evidence": f"{count} of {len(df)} rows are missing this value."
    })

duplicate_count = int(df.duplicated().sum())
if duplicate_count:
    findings.append({
        "type": "Duplicate rows",
        "priority": "Medium",
        "finding": f"{duplicate_count:,} fully duplicated row(s) detected.",
        "evidence": "Rows match across all columns; duplicates may be legitimate or accidental."
    })

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
for col in numeric_cols:
    series = df[col].dropna()
    if len(series) >= 8 and series.nunique() > 1:
        q1, q3 = series.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr > 0:
            lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            mask = (df[col] < lower) | (df[col] > upper)
            count = int(mask.sum())
            if count:
                findings.append({
                    "type": "Potential outliers",
                    "priority": "Medium",
                    "finding": f"{col} has {count:,} value(s) outside the 1.5×IQR range.",
                    "evidence": f"IQR screening bounds: {lower:.4g} to {upper:.4g}. This is a screening rule, not proof of an error."
                })

if findings:
    findings_df = pd.DataFrame(findings)
    display(findings_df)
else:
    findings_df = pd.DataFrame(columns=["type", "priority", "finding", "evidence"])
    print("No basic rule-based findings detected.")

In [ ]:
# Visualize missing values by column
missing_by_column = missing[missing > 0].sort_values(ascending=False)
if len(missing_by_column):
    ax = missing_by_column.plot(kind="bar", figsize=(10, 4), title="Missing values by column")
    ax.set_xlabel("Column")
    ax.set_ylabel("Missing cells")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")

## 6. Machine-learning anomaly scan

In [ ]:
# Choose numeric columns for the scan.
# Edit SELECTED_FEATURES to focus on particular columns.
SELECTED_FEATURES = numeric_cols[:min(5, len(numeric_cols))]
CONTAMINATION = 0.05  # approximate fraction expected to be unusual; tune thoughtfully

if len(SELECTED_FEATURES) == 0:
    print("No numeric columns are available for anomaly detection.")
else:
    features = df[SELECTED_FEATURES].replace([np.inf, -np.inf], np.nan)
    valid_mask = features.notna().all(axis=1)
    valid = features.loc[valid_mask]

    if len(valid) < 10:
        print("At least 10 complete rows are needed for the scan.")
    else:
        scaler = StandardScaler()
        scaled = scaler.fit_transform(valid)
        model = IsolationForest(
            contamination=CONTAMINATION,
            random_state=42,
            n_estimators=200
        )
        labels = model.fit_predict(scaled)
        scores = -model.decision_function(scaled)

        anomaly_results = df.loc[valid_mask].copy()
        anomaly_results["ghosttrace_flag"] = np.where(labels == -1, "Review", "Typical")
        anomaly_results["anomaly_score"] = scores
        anomaly_results = anomaly_results.sort_values("anomaly_score", ascending=False)

        print(f"Rows scanned: {len(valid):,}")
        print(f"Rows flagged for review: {(labels == -1).sum():,}")
        display(anomaly_results.head(20))

## 7. Visualize the anomaly map

In [ ]:
if "anomaly_results" in globals() and len(SELECTED_FEATURES) >= 2:
    x_col, y_col = SELECTED_FEATURES[:2]
    fig, ax = plt.subplots(figsize=(8, 5))
    for label, group in anomaly_results.groupby("ghosttrace_flag"):
        ax.scatter(group[x_col], group[y_col], label=label, alpha=0.75)
    ax.set_title("GHOSTTRACE anomaly map")
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Select at least two numeric features and run the anomaly scan cell first.")

## 8. Explore correlations

In [ ]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(max(6, len(numeric_cols)), max(4, len(numeric_cols) * 0.6)))
    image = ax.imshow(corr, aspect="auto", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(corr.index)))
    ax.set_yticklabels(corr.index)
    ax.set_title("Numeric feature correlation map")
    fig.colorbar(image, ax=ax, label="Correlation")
    plt.tight_layout()
    plt.show()
    display(corr)
else:
    print("At least two numeric columns are needed for correlation analysis.")

## 9. Export findings

In [ ]:
from pathlib import Path

output_dir = Path("ghosttrace_outputs")
output_dir.mkdir(exist_ok=True)

if len(findings_df):
    findings_df.to_csv(output_dir / "ghosttrace_findings.csv", index=False)
if "anomaly_results" in globals():
    anomaly_results.to_csv(output_dir / "ghosttrace_anomaly_scan.csv", index=False)

report_lines = [
    "# GHOSTTRACE Investigation Report",
    "",
    f"- Rows: {len(df)}",
    f"- Columns: {len(df.columns)}",
    f"- Missing cells: {int(df.isna().sum().sum())}",
    f"- Duplicate rows: {int(df.duplicated().sum())}",
    "",
    "## Findings",
]
if len(findings_df):
    for _, row in findings_df.iterrows():
        report_lines.append(f"- [{row['priority']}] {row['type']}: {row['finding']} Evidence: {row['evidence']}")
else:
    report_lines.append("- No basic rule-based findings detected.")
report_lines += [
    "",
    "## Limitations",
    "Automated flags identify records and patterns worth reviewing. They do not prove fraud, error, or causation.",
]
(output_dir / "ghosttrace_report.md").write_text("\n".join(report_lines), encoding="utf-8")

print(f"Saved output files to: {output_dir.resolve()}")
print("Files:", [p.name for p in output_dir.iterdir()])

## 10. Next improvements

Ideas to evolve this into a stronger portfolio project:
1. Add schema-aware contradiction rules (for example, `end_date < start_date`).
2. Add configurable rules for specific domains.
3. Add data-drift comparison between two dataset versions.
4. Build a Streamlit interface around the analysis.
5. Add unit tests and a synthetic test dataset.
6. Add a reproducible audit log of which checks were run and why.

**Responsible use:** Review flagged records before taking action. Outlier detection depends on feature selection and parameter settings.